In [2]:
%cd /home/maia-user/NeuroCBIR/

/home/maia-user/NeuroCBIR


In [3]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np

### Input data
# Path to dataset
DATA_PATH = "/home/maia-user/cifs/Datasets/"

In [8]:
# Load metadata
import pandas as pd
import os

# Your existing list of DataFrames
clinical_ds = [
    pd.read_csv(os.path.join(DATA_PATH, "ADNI", "metadata.csv")),
    pd.read_csv(os.path.join(DATA_PATH, "OASIS3", "metadata.csv")),
    pd.read_csv(os.path.join(DATA_PATH, "AIBL", "metadata.csv")),
    pd.read_csv(os.path.join(DATA_PATH, "SLIM", "metadata.csv")),
    pd.read_csv(os.path.join(DATA_PATH, "MIRIAD", "metadata.csv"))
]

# Combine into a single DataFrame
clinical_ds = pd.concat(clinical_ds, axis=0, ignore_index=True)

# (Optional) check shape and columns
print(clinical_ds.shape)
print(clinical_ds.columns)

(26693, 15)
Index(['GUID', 'project', 'subject', 'timepoint', 'scan_type',
       'field_strength', 'manufacturer', 'model_name', 'disease', 'age',
       'brain', 'sex', 'acq_date', 'raw', 'seg'],
      dtype='object')


In [14]:
clinical_ds.manufacturer.unique()

array([nan, 'GE MEDICAL SYSTEMS', 'Philips Medical Systems', 'SIEMENS',
       'Philips Healthcare', 'Siemens'], dtype=object)

In [15]:
disease_mapping = {
    "CN": "CN", "Normal": "CN", "Control": "CN",
    "MCI": "MCI", "Mild Cognitive Impairment": "MCI", "EMCI": "MCI", "LMCI": "MCI", "SMC": "MCI",
    "AD": "AD", "Alzheimer's Disease": "AD"
}

manufacturer_mapping = {
    "Siemens": "SH", "Siemens Healthineers": "SH", "SIEMENS": "SH",
    "GE": "GE", "GE MEDICAL SYSTEMS": "GE"
}


import pandas as pd

# Map disease categories
clinical_ds['disease_cat'] = clinical_ds['disease'].map(disease_mapping).fillna("Other")

# Map manufacturers
clinical_ds['manufacturer_cat'] = clinical_ds['manufacturer'].map(manufacturer_mapping).fillna("Other")

# Function to compute summary per project
def summarize_project(df):
    return pd.Series({
        "NI": len(df),  # total scans
        "NS": df['subject'].nunique(),  # unique subjects
        "Age": f"{df['age'].mean():.1f} ± {df['age'].std():.1f}",  # mean ± std
        "CN": (df['disease_cat'] == "CN").sum(),
        "MCI": (df['disease_cat'] == "MCI").sum(),
        "AD": (df['disease_cat'] == "AD").sum(),
        "Other": (df['disease_cat'] == "Other").sum(),
        "1.5T": (df['field_strength'] == 1.5).sum(),
        "3T": (df['field_strength'] == 3).sum(),
        "SH": (df['manufacturer_cat'] == "SH").sum(),
        "GE": (df['manufacturer_cat'] == "GE").sum(),
        "Other_manufacturer": (df['manufacturer_cat'] == "Other").sum()
    })

# Group and summarize
summary_table = clinical_ds.groupby("project").apply(summarize_project).reset_index()

# Rename columns for display
summary_table.columns = [
    "Dataset", "NI", "NS", "Age", "CN", "MCI", "AD", "Other", "1.5T", "3T", "SH", "GE", "Other"
]

print(summary_table)



  Dataset     NI    NS         Age    CN    MCI    AD  Other  1.5T    3T  \
0    adni  20971  2422  73.6 ± 7.1  6360  11926  2670     15  8114  9101   
1    aibl   1296   691  73.6 ± 6.7   947    184   158      7   269  1027   
2  miriad    708    69  69.7 ± 6.9   243      0   465      0   708     0   
3  oasis3   2681  1316  70.5 ± 9.1  2183    395   102      1   212  2464   
4    slim   1037   588  20.7 ± 1.4  1037      0     0      0     0  1037   

     SH    GE  Other  
0  7587  3662   9722  
1  1296     0      0  
2     0   708      0  
3  2681     0      0  
4  1037     0      0  


/tmp/ipykernel_188469/3644031043.py:39: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary_table = clinical_ds.groupby("project").apply(summarize_project).reset_index()
